# C10 prereg43：V2 主幹 320,000 步、種子 0（Colab G4）

執行階段選 G4（RTX PRO 6000），按「執行階段 → 全部執行」。套件沿用 `C10/p42/C10_p42_kit.zip`（prereg42），
1b 格寫入 prereg43 的包裝程式；輸出在 `C10/p43/`。斷線後重按「全部執行」會從最後存檔接續。

In [ ]:
# 0. 把 Drive 上 C10/p42/kit_parts/ 的 66 份接回 C10_p42_kit.zip（已存在且雜湊對就略過）
import hashlib, os, glob
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/C10/p42'
KIT = os.path.join(D, 'C10_p42_kit.zip')
SHA = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()
if not (os.path.exists(KIT) and sha(KIT) == SHA):
    parts = sorted(glob.glob(os.path.join(D, 'kit_parts', 'kit.part*')))
    assert len(parts) == 66, 'STOP: kit_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p42_kit.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == SHA, 'STOP: 接回的 zip 雜湊不對'
    import shutil; shutil.copyfile(tmp, KIT)
    assert sha(KIT) == SHA, 'STOP: 複製到 Drive 後雜湊不對'
print('kit OK', SHA[:12])


In [ ]:
# 1. 雲端硬碟、GPU、套件（不用改）
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE, WORK, VARIANT = '/content/drive/MyDrive/C10/p42', '/content/c10', 'V2'
KIT_SHA, FILES = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306', {"data/prereg06/feats/meta.npz": "b1b5def7312dc6321ab08dd7e74e9b9069c49e9efb8caf7133b9869bf7338df5", "data/prereg36/cache/cache.json": "764a61937e60ab660972ecbe8a585e4205bd27a132ce74a64f4cc7390533c48a", "data/prereg36/cache/codes.npy": "be5831abbccf22be3c96725082939638e6607f88ad45b1ed2164db33060c0303", "data/prereg36/cache/meta.npz": "fa8bf685be64aa100746238caeaa87f917d6adc194d2026d83c7865f61f834d3", "data/prereg36/cache/off.npy": "7c352279359c5035c1693b78a783d534c8b6ac1e7f870e56655b4b764fb2eaf8", "results/prereg36/explore/p36_embed.py": "2f2e732d1309e25e0b70e370583c07b74dd4c692c6490db8fe1b533ecda32c21", "results/prereg36/explore/p36_lib.py": "8df27b37a6b3de786b54af25908d3d91ffe4c030dceb7350d9dd130a14ce241f", "results/prereg36/explore/p36_train.py": "8abc5ef22eba35b70be06c624f7071218d0a4266affe5494d1207f8819271211", "results/prereg36/explore/split_record.json": "2c847257139cd73b9a1808c003ba30b86bcd59eb6860edef8043afc82535e7e3", "results/prereg42/explore/p42_embed.py": "23b60a6f71d1aad5881acd10c87ef2c31a9ed81e6ac2d369c3d29615452b69fd", "results/prereg42/explore/p42_lib.py": "b5045f8b8222e1c3e6891675eb4b3296fe4254b974421c71853ea8a43cea6eea", "results/prereg42/explore/p42_train.py": "ec5d7baf7bf16753cd9d0fb7369e3ff4cb5a9a271d8ed28cf264691bb2359c80", "tools/c10data.py": "2906d97fe41171dc955288d0a6e695fcef905417d18d94cb98f146aac74c6495"}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout)
tv = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()
print('torch', tv)   # in a subprocess: the notebook itself never holds the GPU (p42_train refuses to share it)
assert tv.endswith('True'), 'STOP: 沒有 GPU，請在「變更執行階段類型」選 GPU'
kit = os.path.join(DRIVE, 'C10_p42_kit.zip')
assert os.path.exists(kit), 'STOP: 找不到 ' + kit
ok_flag = os.path.join(WORK, '.kit_ok')
if not (os.path.exists(ok_flag) and open(ok_flag).read() == KIT_SHA):
    assert sha(kit) == KIT_SHA, 'STOP: 套件檔的 SHA-256 不對'
    with zipfile.ZipFile(kit) as z:
        z.extractall(WORK)
    for rel, h in FILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(ok_flag, 'w').write(KIT_SHA)
print('kit ok', len(FILES), 'files')
os.makedirs(os.path.join(DRIVE, VARIANT), exist_ok=True)

In [ ]:
# 1b. prereg43 的包裝程式 p43_run.py（只設 320,000 步、每 32,000 步存檔、種子）；核雜湊後寫進工作目錄（不用改）
import base64
H, B64 = 'a767ad139e921de3ff5fe2afe48165c048efa2afdf944bc8402e9ec300f98064', 'IiIicHJlcmVnNDM6IHRoZSBwcmVyZWc0MiBWMiBjb2RlIHdpdGggYSAzMjAsMDAwLXN0ZXAgc2NoZWR1bGUgYW5kIGEgY2hvc2VuIHNlZWQgKFBMQU4ubWQgMSkuCgogICAgR1BVUFkgcmVzdWx0cy9wcmVyZWc0My9leHBsb3JlL3A0M19ydW4ucHkgdHJhaW4gLS1zZWVkIFMgW3A0Ml90cmFpbi5weSBhcmd1bWVudHM7IC0tdmFyaWFudCBpcyBmaXhlZCB0byBWMl0KICAgIEdQVVBZIHJlc3VsdHMvcHJlcmVnNDMvZXhwbG9yZS9wNDNfcnVuLnB5IGVtYmVkIFtwNDJfZW1iZWQucHkgYXJndW1lbnRzXQoKTm90aGluZyBvZiByZXN1bHRzL3ByZXJlZzQyL2V4cGxvcmUgaXMgZWRpdGVkOiB0aGlzIHdyYXBwZXIgb25seSBzZXRzIHRocmVlIGNvbnN0YW50cyBvZiBwNDJfbGliIGJlZm9yZSBoYW5kaW5nIG92ZXIuCiIiIgppbXBvcnQgb3MKaW1wb3J0IHN5cwoKSEVSRSA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKQpSRVBPID0gb3MucGF0aC5hYnNwYXRoKG9zLnBhdGguam9pbihIRVJFLCAiLi4iLCAiLi4iLCAiLi4iKSkKUDQyID0gb3MucGF0aC5qb2luKFJFUE8sICJyZXN1bHRzIiwgInByZXJlZzQyIiwgImV4cGxvcmUiKQpzeXMucGF0aC5pbnNlcnQoMCwgUDQyKQppbXBvcnQgcDQyX2xpYiBhcyBWICAjIG5vcWE6IEU0MDIKClRPVEFMX1NURVBTID0gMzIwMDAwCkNLUFRfRVZFUlkgPSAzMjAwMApTRUVEUyA9ICgwLCAxKQpWQVJJQU5UID0gIlYyIgoKCmRlZiBtYWluKGFyZ3Y9Tm9uZSk6CiAgICBhcmd2ID0gbGlzdChzeXMuYXJndlsxOl0gaWYgYXJndiBpcyBOb25lIGVsc2UgYXJndikKICAgIGlmIG5vdCBhcmd2IG9yIGFyZ3ZbMF0gbm90IGluICgidHJhaW4iLCAiZW1iZWQiKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJ1c2FnZTogcDQzX3J1bi5weSB0cmFpbiAtLXNlZWQgUyAuLi4gfCBlbWJlZCAuLi4iKQogICAgVi5UT1RBTF9TVEVQUywgVi5DS1BUX0VWRVJZID0gVE9UQUxfU1RFUFMsIENLUFRfRVZFUlkKICAgIGlmIGFyZ3ZbMF0gPT0gInRyYWluIjoKICAgICAgICByZXN0ID0gYXJndlsxOl0KICAgICAgICBpZiAiLS1zZWVkIiBub3QgaW4gcmVzdDoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS1zZWVkIGlzIHJlcXVpcmVkIikKICAgICAgICBpID0gcmVzdC5pbmRleCgiLS1zZWVkIikKICAgICAgICBzZWVkID0gaW50KHJlc3RbaSArIDFdKQogICAgICAgIGlmIHNlZWQgbm90IGluIFNFRURTOgogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJzZWVkIG11c3QgYmUgb25lIG9mICVzIiAlIChTRUVEUywpKQogICAgICAgIGRlbCByZXN0W2k6aSArIDJdCiAgICAgICAgaWYgIi0tdmFyaWFudCIgaW4gcmVzdDoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS12YXJpYW50IGlzIGZpeGVkIHRvICVzIiAlIFZBUklBTlQpCiAgICAgICAgVi5TRUVEID0gc2VlZAogICAgICAgIGltcG9ydCBwNDJfdHJhaW4gYXMgVAogICAgICAgIGlmIFQuViBpcyBub3QgVjoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgicDQyX3RyYWluIGRpZCBub3QgaW1wb3J0IHRoZSBzYW1lIHA0Ml9saWIgbW9kdWxlIikKICAgICAgICByZXR1cm4gVC5tYWluKFsiLS12YXJpYW50IiwgVkFSSUFOVF0gKyByZXN0KQogICAgaW1wb3J0IHA0Ml9lbWJlZCBhcyBFCiAgICByZXR1cm4gRS5tYWluKGFyZ3ZbMTpdKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg=='
data = base64.b64decode(B64)
assert hashlib.sha256(data).hexdigest() == H, 'STOP: 內嵌的 p43_run.py 不對'
os.makedirs(os.path.join(WORK, 'results/prereg43/explore'), exist_ok=True)
with open(os.path.join(WORK, 'results/prereg43/explore/p43_run.py'), 'wb') as f:
    f.write(data)
assert sha(os.path.join(WORK, 'results/prereg43/explore/p43_run.py')) == H
P43 = '/content/drive/MyDrive/C10/p43'
os.makedirs(P43, exist_ok=True)
print('overlay ok', H[:12])

In [ ]:
# 2. 訓練 V2 320,000 步、種子 0（約 20 小時；每 32,000 步存到 C10/p43/S0/；斷線後重按「全部執行」接續）
out = os.path.join(P43, 'S0')
os.makedirs(out, exist_ok=True)
rec = os.path.join(out, 'train_record_S0.json')
workers = str(max(2, min(12, (os.cpu_count() or 4) - 1)))
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', PYTHONIOENCODING='utf-8')
def run(args, log):
    with open(log, 'a') as lf:
        p = subprocess.Popen([sys.executable, '-u'] + args, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        return p.wait()
done = os.path.exists(rec) and json.load(open(rec)).get('finished')
if not done:
    rc = run(['results/prereg43/explore/p43_run.py', 'train', '--seed', '0', '--workers', workers, '--auto-resume', '--out', out, '--record', rec, '--min-free-ram-mb', '2048', '--log-every', '1000'], os.path.join(out, 'train_log.txt'))
    assert rc == 0, 'STOP: 訓練結束碼 %d（4＝記憶體不足或 STOP 檔，重按「全部執行」會接續）' % rc
print('training finished')

In [ ]:
# 3. D 玩家的向量（最後存檔的 raw、EMA；不用改）
ck = os.path.join(out, 'ckpt', 'step_320000.pt')
emb_root = os.path.join(P43, 'emb_S0')
for w in ('raw', 'ema'):
    if os.path.exists(os.path.join(emb_root, 'V2_step_320000_%s_tta1' % w, 'store_D.npz')):
        continue
    rc = run(['results/prereg43/explore/p43_run.py', 'embed', '--ckpt', ck, '--players', 'D', '--workers', workers, '--emb-root', emb_root, '--weights', w], os.path.join(out, 'embed_log.txt'))
    assert rc == 0, 'STOP: 向量結束碼 %d' % rc
print('embeddings finished')

In [ ]:
# 4. 打包（不用改）
res = os.path.join(P43, 'C10_p43_S0_results.zip')
with zipfile.ZipFile(res + '.tmp', 'w', zipfile.ZIP_STORED) as z:
    for name in ('train_record_S0.json', 'train_log.txt', 'train_log.jsonl', 'embed_log.txt'):
        p = os.path.join(out, name)
        if os.path.exists(p):
            z.write(p, 'S0/%s' % name)
    z.write(ck, 'S0/ckpt/step_320000.pt')
    for d in sorted(os.listdir(emb_root)):
        z.write(os.path.join(emb_root, d, 'store_D.npz'), 'emb_S0/%s/store_D.npz' % d)
os.replace(res + '.tmp', res)
drive.flush_and_unmount()
print('完成', os.path.basename(res))